In [1]:
#NBA.COM TEAM PAGES
from dataclasses import dataclass
import json
import time
import random
import pandas as pd
from bs4 import BeautifulSoup
from typing import Callable, Any
from requests import Response
import requests


In [2]:
import re
import unicodedata

SUFFIX_RE = re.compile(r'[\s,]+(jr|sr|ii|iii|iv|v)\.?$', re.IGNORECASE)

def clean_name(s):
    if not isinstance(s, str):
        return s
    try:
        s = s.encode('latin1').decode('utf-8')  # fix mojibake, if present
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    normalized = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in normalized if not unicodedata.combining(c))
    s = SUFFIX_RE.sub('', s.strip())
    return s.strip()

In [3]:
team_dim = pd.read_csv('/Users/austinriley/bball_analytics/NBA_draft_pick_value_model/team_dim.csv')

In [4]:
#DEFINE BBREF SCRAPER
class HTMLScraper:
    """Class for scraping the web"""

    proxy: str | None = None
    timeout: int = 10  # seconds to wait before giving up on a request
    max_retries: int = 3
    retry_backoff: int = 15  # base seconds to wait before retrying

    def scrape(self, url: str, parser_fun: Callable) -> Any:
        page = self.get_page(url)
        soup = BeautifulSoup(page.text, "html.parser")
        return parser_fun(soup)

    def get_page(self, url: str) -> Response:
        """
        Sends a GET request with a timeout, and retries with backoff
        if rate-limited or blocked (429/403) or the connection fails.
        """

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        }

        for attempt in range(1, self.max_retries + 1):
            try:
                with requests.Session() as session:
                    page = session.get(
                        url,
                        proxies=self._proxies(),
                        headers=headers,
                        timeout=self.timeout,
                    )

                if page.status_code in (429, 403):
                    wait = int(
                        page.headers.get("Retry-After", self.retry_backoff * attempt)
                    )
                    print(f"Blocked ({page.status_code}) on {url}, waiting {wait}s (attempt {attempt})")
                    time.sleep(wait)
                    continue

                page.raise_for_status()
                return page

            except requests.exceptions.RequestException as e:
                print(f"Request failed for {url}: {e} (attempt {attempt})")
                if attempt == self.max_retries:
                    raise
                time.sleep(self.retry_backoff * attempt)

        raise RuntimeError(f"Exhausted retries fetching {url}")

    def _proxies(self) -> dict | None:
        """If the object is initialized with a proxy then use it"""
        if self.proxy:
            return {"https": self.proxy, "http": self.proxy}
        return None


In [12]:

#NBA . COM SCRAPER
# nba.com slugs that aren't just the last word of the team name, lowercased
SLUG_OVERRIDES = {
    "Portland Trail Blazers": "blazers",
    "Philadelphia 76ers": "sixers",
}

# defunct franchise rows in team_dim
DEFUNCT_ALIASES = {"NOH", "NJN"}


@dataclass
class NBATeamPageScraper(HTMLScraper):
    """
    Scrapes nba.com/team/{team_id}/{slug}. The page is a Next.js app, so the
    data lives in the __NEXT_DATA__ JSON blob rather than the visible HTML.
    """

    base_url: str = "https://www.nba.com"

    def get_page_data(self, team_id: int, slug: str) -> dict:
        """Returns props.pageProps, the part of __NEXT_DATA__ holding the content."""
        url = f"{self.base_url}/team/{team_id}/{slug}"
        return self.scrape(url, self._parse_next_data)

    def debug_keys(self, team_id: int, slug: str, max_depth: int = 2) -> None:
        """Prints the JSON structure so you can see what's available."""
        data = self.get_page_data(team_id, slug)

        def _walk(obj, path="pageProps", depth=0):
            if depth > max_depth:
                return
            if isinstance(obj, dict):
                for k, v in obj.items():
                    kind = type(v).__name__
                    size = f"[{len(v)}]" if isinstance(v, (list, dict)) else ""
                    print("  " * depth + f"{path}.{k}: {kind}{size}")
                    _walk(v, f"{path}.{k}", depth + 1)

        _walk(data)

    def get_team(self, team_id: int, slug: str) -> dict:
        """Returns {'roster': [...], 'coaches': [...], 'team_info': {...}}."""
        data = self.get_page_data(team_id, slug)

        roster = self._find_record_list(data, "PLAYER_ID")
        coaches = self._find_record_list(data, "COACH_ID") or self._find_record_list(data, "COACH_NAME")
        team_info = data.get("team") if isinstance(data.get("team"), dict) else {}

        for rec in roster:
            rec["team_id"] = team_id
        for rec in coaches:
            rec["team_id"] = team_id

        # keep only flat (scalar) fields from team_info so it fits in a row
        flat_info = {k: v for k, v in team_info.items() if not isinstance(v, (dict, list))}
        flat_info["team_id"] = team_id

        return {"roster": roster, "coaches": coaches, "team_info": flat_info}

    # private functions

    @staticmethod
    def _parse_next_data(page: BeautifulSoup) -> dict:
        script = page.find("script", id="__NEXT_DATA__")
        if script is None or not script.string:
            raise ValueError(
                "No __NEXT_DATA__ script found. nba.com may have blocked the "
                "request or served a different page."
            )
        return json.loads(script.string).get("props", {}).get("pageProps", {})

    @classmethod
    def _find_record_list(cls, obj, marker_key: str) -> list[dict]:
        """
        Recursively finds the first list of dicts whose records contain
        marker_key (e.g. PLAYER_ID for the roster), so it doesn't break if
        nba.com moves things around in the JSON.
        """
        if isinstance(obj, list):
            if obj and all(isinstance(x, dict) for x in obj) and marker_key in obj[0]:
                return [dict(x) for x in obj]
            for item in obj:
                found = cls._find_record_list(item, marker_key)
                if found:
                    return found
        elif isinstance(obj, dict):
            for v in obj.values():
                found = cls._find_record_list(v, marker_key)
                if found:
                    return found
        return []


def team_slug(team_name: str) -> str:
    return SLUG_OVERRIDES.get(team_name, team_name.split(" ")[-1].lower())


# ============================================================
# RUN IT FOR ALL 30 TEAMS
# ============================================================

team_page_scraper = NBATeamPageScraper()
current_teams = team_dim[~team_dim["team_alias"].isin(DEFUNCT_ALIASES)]

all_roster, all_coaches, all_info, failed = [], [], [], []

for i, t in enumerate(current_teams.itertuples(), 1):
    slug = team_slug(t.team)
    print(f"\r{i}/{len(current_teams)}: {t.team_alias}", end="", flush=True)
    try:
        result = team_page_scraper.get_team(t.team_id, slug)
        all_roster.extend(result["roster"])
        all_coaches.extend(result["coaches"])
        all_info.append(result["team_info"])
        if not result["roster"]:
            print(f"\n⚠️ No roster found for {t.team_alias} — run debug_keys({t.team_id}, '{slug}')")
    except Exception as e:
        print(f"\n⚠️ Failed {t.team_alias}: {type(e).__name__}: {e}")
        failed.append((t.team_alias, str(e)))
    time.sleep(random.uniform(3, 5))

nba_roster_df = pd.DataFrame(all_roster)
nba_coaches_df = pd.DataFrame(all_coaches)
nba_team_info_df = pd.DataFrame(all_info)

# join keys matching the rest of the notebook
nba_roster_df = pd.merge(nba_roster_df, team_dim[["team_id", "team_alias"]], how="left", on="team_id")
if "PLAYER" in nba_roster_df:
    nba_roster_df["player"] = (
        nba_roster_df["PLAYER"].apply(clean_name).str.lower().str.replace(".", "", regex=False)
    )

print(f"\n\nRoster rows: {len(nba_roster_df)} | Coaches: {len(nba_coaches_df)} | Teams: {len(nba_team_info_df)}")
if failed:
    print(f"{len(failed)} teams failed: {failed}")

# nba_roster_df.to_csv("NBA_com_rosters_2026-27.csv", index=False)

nba_roster_df.head()


30/30: CHA

Roster rows: 523 | Coaches: 299 | Teams: 30


,TeamID,SEASON,LeagueID,PLAYER,NICKNAME,PLAYER_SLUG,NUM,POSITION,HEIGHT,WEIGHT,BIRTH_DATE,AGE,EXP,SCHOOL,PLAYER_ID,HOW_ACQUIRED,SUPPLEMENTAL_STATUS,team_id,team_alias,player
0,1610612737,2026,00,Luguentz Dort,Luguentz,luguentz-dort,0,G,6-4,220,"APR 19, 1999",27,7,Arizona State,1629652,Traded from OKC on 07/19/26,0,1610612737,ATL,luguentz dort
1,1610612737,2026,00,RayJ Dennis,RayJ,rayj-dennis,00,G,6-1,180,"MAR 30, 2001",25,2,Baylor,1642484,Signed on 12/31/25,1,1610612737,ATL,rayj dennis
2,1610612737,2026,00,Keshon Gilbert,Keshon,keshon-gilbert,00,G,6-3,200,"JUN 30, 2003",23,1,Iowa State,1642933,Signed on 03/04/26,1,1610612737,ATL,keshon gilbert
3,1610612737,2026,00,Jalen Johnson,Jalen,jalen-johnson,1,F,6-8,219,"DEC 18, 2001",24,5,Duke,1630552,#20 Pick in 2021 Draft,0,1610612737,ATL,jalen johnson
4,1610612737,2026,00,Aaron Wiggins,Aaron,aaron-wiggins,2,G,6-5,190,"JAN 02, 1999",27,5,Maryland,1630598,Traded from OKC on 07/06/26,0,1610612737,ATL,aaron wiggins


In [9]:
#BBREF SCRAPER

from dataclasses import dataclass
from bs4 import BeautifulSoup, Comment


@dataclass
class BaskRefPlayerStatsScraper(HTMLScraper):
    """Scrapes league-wide per-player stat tables (advanced, shooting, etc.)."""

    base_url: str = "https://www.basketball-reference.com"

    STAT_TYPE_CONFIG = {
        "advanced": {
            "url_suffix": "advanced",
            "table_ids": ["advanced", "advanced_stats"],
        },
        "shooting": {
            "url_suffix": "shooting",
            "table_ids": ["shooting", "shooting_stats"],
        },
        "per_game": {
            "url_suffix": "per_game",
            "table_ids": ["per_game_stats", "per_game"],
        },
    }

    def get_stats(self, season_year: int, stat_type: str) -> list[dict]:
        config = self.STAT_TYPE_CONFIG[stat_type]
        url = f"{self.base_url}/leagues/NBA_{season_year}_{config['url_suffix']}.html"
        return self.scrape(url, lambda page: self._parse_stats_table(page, config["table_ids"]))

    def debug_list_tables(self, season_year: int, stat_type: str) -> dict:
        config = self.STAT_TYPE_CONFIG[stat_type]
        url = f"{self.base_url}/leagues/NBA_{season_year}_{config['url_suffix']}.html"

        def _list_ids(page: BeautifulSoup) -> dict:
            direct_ids = [t.get("id") for t in page.find_all("table")]
            comment_ids = []
            for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
                if "<table" in comment:
                    inner = BeautifulSoup(comment, "html.parser")
                    comment_ids.extend(t.get("id") for t in inner.find_all("table"))
            return {"direct": direct_ids, "in_comments": comment_ids}

        return self.scrape(url, _list_ids)

    def _parse_stats_table(self, page: BeautifulSoup, table_ids: list[str]) -> list[dict]:
        table = None
        for tid in table_ids:
            table = page.select_one(f"#{tid}")
            if table is not None:
                break

        if table is None:
            for tid in table_ids:
                table = self._find_table_in_comments(page, tid)
                if table is not None:
                    break

        if table is None:
            raise ValueError(
                f"Could not locate a stats table with any of ids {table_ids}. "
                "Run debug_list_tables() to see available table ids."
            )

        rows = []
        for tr in table.select("tbody tr"):
            if tr.get("class") and "thead" in tr.get("class"):
                continue
            row = {}
            for cell in tr.select("td, th"):
                stat = cell.get("data-stat")
                if stat:
                    row[stat] = cell.text.strip()
            if row:
                rows.append(row)
        return rows

    @staticmethod
    def _find_table_in_comments(page: BeautifulSoup, id_substring: str):
        for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in comment and id_substring in comment:
                inner = BeautifulSoup(comment, "html.parser")
                found = inner.select_one(f"table[id*={id_substring}]")
                if found is not None:
                    return found
        return None

In [25]:
stats_scraper = BaskRefPlayerStatsScraper()

per_game_df = pd.DataFrame(stats_scraper.get_stats(2027, 'per_game'))
per_game_df['season'] = 2027

# drop the League Average row
per_game_df = per_game_df[per_game_df['name_display'] != 'League Average']

# traded players: keep the 2TM/3TM total row, drop the individual team rows
multi_team_labels = {'2TM', '3TM', '4TM', '5TM'}
has_agg = per_game_df.groupby('name_display')['team_name_abbr'].transform(
    lambda s: s.isin(multi_team_labels).any()
)
per_game_df = per_game_df[~(has_agg & ~per_game_df['team_name_abbr'].isin(multi_team_labels))].copy()

# numeric columns
non_numeric = {'name_display', 'team_name_abbr', 'pos', 'awards'}
for col in per_game_df.columns.difference(non_numeric):
    per_game_df[col] = pd.to_numeric(per_game_df[col], errors='coerce')

per_game_df['name_display'] = (
    per_game_df['name_display'].apply(clean_name).str.lower().str.replace('.', '', regex=False)
)
# per_game_df.to_csv('NBA_per_game_2027.csv', index=False)
per_game_df.loc[per_game_df['team_name_abbr'] == 'BRK', 'team_name_abbr'] = 'BKN'
per_game_df.loc[per_game_df['team_name_abbr'] == 'CHO', 'team_name_abbr'] = 'CHA'
per_game_df.loc[per_game_df['team_name_abbr'] == 'PHO', 'team_name_abbr'] = 'PHX'
nba_roster_df.loc[nba_roster_df['team_alias'] == 'BRK', 'team_alias'] = 'BKN'
nba_roster_df.loc[nba_roster_df['team_alias'] == 'CHO', 'team_alias'] = 'CHA'
nba_roster_df.loc[nba_roster_df['team_alias'] == 'PHO', 'team_alias'] = 'PHX'

roster_mpg = pd.merge(nba_roster_df[['player','team_alias']],per_game_df[['name_display','mp_per_g','team_name_abbr']],how='left',left_on=['player','team_alias'],right_on=['name_display','team_name_abbr'])
roster_mpg['team_mpg_rk'] = roster_mpg.groupby('team_alias')['mp_per_g'].rank(method='first',ascending=False)
roster_mpg

,player,team_alias,name_display,mp_per_g,team_name_abbr,team_mpg_rk
0,luguentz dort,ATL,luguentz dort,23.0,ATL,7.0
1,rayj dennis,ATL,NaN,NaN,NaN,NaN
2,keshon gilbert,ATL,NaN,NaN,NaN,NaN
3,jalen johnson,ATL,jalen johnson,33.2,ATL,1.0
4,aaron wiggins,ATL,aaron wiggins,26.4,ATL,5.0
...,...,...,...,...,...,...
518,kylan boswell,CHA,NaN,NaN,NaN,NaN
519,hannes steinbach,CHA,hannes steinbach,12.0,CHA,10.0
520,brandon miller,CHA,brandon miller,33.8,CHA,1.0
521,tidjane salaun,CHA,tidjane salaun,15.0,CHA,9.0
